# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/viki22uied/ML/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Question:** among thousands of existing content pages, which ones should an editor review
first for a refresh? **Decision supported:** weekly/monthly editorial triage — where to spend
limited human review time across a large content portfolio. Full framing in
`work/capstone_report.md`, section 1.


In [1]:
print('See markdown above -- framing only, no computation needed here.')

See markdown above -- framing only, no computation needed here.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Starter dataset: `data/raw/content_refresh_anonymized.csv` — 30,000 pseudonymized content items,
32 clients, trailing-90-day metrics. Earlier modeling stage (ML-08) additionally used a
`month=2026-03` cut of the gated warehouse release. `trend_direction`/`trend_pct` (label-derived)
and `client_id`/`content_id` (grouping only) are never model features. Details in
`work/capstone_report.md`, section 2.


In [2]:
import sys
sys.path.append('../../scripts')
import pandas as pd
from ml_utils import RAW_PATH

raw = pd.read_csv(RAW_PATH)
print(f"Rows: {len(raw):,}   Clients: {raw['client_id'].nunique()}   Columns: {raw.shape[1]}")
print(f"Excluded-as-features (label-derived): trend_direction, trend_pct")
print(f"ID columns (grouping only, never features): client_id, content_id")


Rows: 30,000   Clients: 32   Columns: 44
Excluded-as-features (label-derived): trend_direction, trend_pct
ID columns (grouping only, never features): client_id, content_id


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label:** `is_declining_label = (trend_direction == "down")` (reference-pipeline stage) /
`needs_review_label` = bottom-30th-percentile March CTR (warehouse stage — ML-08). **Baseline:**
a transparent weighted rule (visibility + freshness + position opportunity + depth gap), never a
model. **Model:** Random Forest, compared against Logistic Regression and a Decision Tree.
**Validation:** client-grouped split in every stage — no client's pages appear in both train and
test. **Leakage checks:** re-run in `work/notebooks/w06_validation_audit.ipynb` section 3 —
confirmed no label-derived, future-window, or decision-derived features in the final feature set.
Full detail: `work/capstone_report.md`, sections 3–4.


In [3]:
print('See markdown above -- methodology summary, backed by w04-w07 notebooks.')

See markdown above -- methodology summary, backed by w04-w07 notebooks.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*


In [4]:
import sys, json
sys.path.append('../../scripts')
import pandas as pd
from ml_utils import OUTPUT_DIR

results = json.loads((OUTPUT_DIR / "model_results.json").read_text())
comparison = pd.DataFrame({
    "stage": ["warehouse (ML-08)", "warehouse (ML-08)", "starter (reference pipeline)", "starter (reference pipeline)",
              "starter (w06 before -- naive random split)", "starter (w06 after -- honest client split)"],
    "split": ["client-grouped", "client-grouped", "client-holdout", "client-holdout", "random (naive)", "client-grouped (honest)"],
    "model": ["random_forest", "baseline_rule", "random_forest", "baseline_rule", "random_forest", "random_forest"],
    "auc": [0.846, 0.684, 0.750, 0.627, 0.7598, 0.6058],
    "p_at_50": [0.98, 0.08, 0.740, 0.240, 0.92, 0.56],
})
print(comparison.to_string(index=False))
print()
print("Base rates -- warehouse label (needs_review_label): 0.431; starter label (is_declining_label): 0.542 "
      "(from outputs/model_results.json 'target_positive_rate').")
print(f"Reference-pipeline target_positive_rate on disk: {results.get('target_positive_rate', 'see outputs/model_results.json')}")


                                     stage                   split         model    auc  p_at_50
                         warehouse (ML-08)          client-grouped random_forest 0.8460     0.98
                         warehouse (ML-08)          client-grouped baseline_rule 0.6840     0.08
              starter (reference pipeline)          client-holdout random_forest 0.7500     0.74
              starter (reference pipeline)          client-holdout baseline_rule 0.6270     0.24
starter (w06 before -- naive random split)          random (naive) random_forest 0.7598     0.92
starter (w06 after -- honest client split) client-grouped (honest) random_forest 0.6058     0.56

Base rates -- warehouse label (needs_review_label): 0.431; starter label (is_declining_label): 0.542 (from outputs/model_results.json 'target_positive_rate').
Reference-pipeline target_positive_rate on disk: 0.5420666666666667


## 5. Limitations

*What this work cannot claim.*

- Association, not causation: a high decline-risk score describes a resemblance to pages that
  already declined, not proof that refreshing a page will reverse its trend.
- Build-window bound: both stages reflect one dataset snapshot each; no time-forward validation
  (prior 90 days → next 30 days) exists yet — noted as the natural next step.
- Split-to-split variance is real on a 32-client dataset: an independent client-grouped re-run
  in this report scored AUC 0.606 vs. the reference pipeline's own 0.750 client-holdout number —
  a caution against treating any single holdout as precise.
- No client-specific tuning; one global model may under- or over-rank an unusual client's content
  mix. Full list: `work/capstone_report.md`, section 7 and `w07_action_playbook.ipynb` section 2.


In [5]:
print('See markdown above.')

See markdown above.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*


In [6]:
import sys
sys.path.append('../../scripts')
import numpy as np
import pandas as pd
from ml_utils import PROCESSED_DIR

predictions = pd.read_csv(PROCESSED_DIR / "model_predictions.csv")
baseline_queue = pd.read_csv(PROCESSED_DIR / "baseline_refresh_queue.csv")
queue = predictions.merge(
    baseline_queue[["content_id", "reason_codes"]], on="content_id", how="left"
).sort_values("prob_random_forest", ascending=False).reset_index(drop=True)

declining_rate_top50 = queue.head(50)["is_declining_label"].mean()
declining_rate_all = queue["is_declining_label"].mean()
print(f"Declining-label rate, top-50 of the ranked action queue: {declining_rate_top50:.3f}")
print(f"Declining-label rate, full portfolio (base rate): {declining_rate_all:.3f}")
print(f"Lift: {declining_rate_top50 / declining_rate_all:.2f}x")
print()
print("Top reason codes in the top-200 of the queue:")
print(queue.head(200)["reason_codes"].str.split("|").explode().value_counts().head(6))


Declining-label rate, top-50 of the ranked action queue: 1.000
Declining-label rate, full portfolio (base rate): 0.542
Lift: 1.84x

Top reason codes in the top-200 of the queue:
reason_codes
declining_with_demand          198
low_ctr_visible_page            58
low_engagement_visible_page     47
page_one_decay_risk              2
general_refresh_review           2
Name: count, dtype: int64


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*


In [7]:
import sys
sys.path.append('../../scripts')
from ml_utils import simple_svg_bar_chart
from pathlib import Path

labels = ["Warehouse RF\n(client-grouped)", "Warehouse baseline", "Starter RF\n(client-holdout)",
          "Starter baseline", "w06 naive split\n(leaky)", "w06 honest split"]
values = [0.846, 0.684, 0.750, 0.627, 0.7598, 0.6058]

out_path = Path("../figures/capstone_auc_comparison.svg")
out_path.parent.mkdir(parents=True, exist_ok=True)
simple_svg_bar_chart("ROC AUC across stages and splits", labels, values, out_path)
print(f"Wrote chart: {out_path}")


Wrote chart: ..\figures\capstone_auc_comparison.svg


## 8. Five-minute demo, social cut, employer summary (ML-12)

**5-minute demo outline:**
1. (30s) The question: which of 30,000+ pages should an editor look at first?
2. (60s) Show the baseline rule and its Precision@50 (0.240 / 0.08) — simple, transparent, but
   leaves a lot of signal on the table.
3. (90s) Show the model beating it on the same client-grouped split (0.740 / 0.98 Precision@50
   depending on stage) — and immediately show the w06 "before vs. after" split comparison to
   prove the number isn't just leakage (naive split 0.92 vs. honest split 0.56 Precision@50).
4. (60s) Walk the top of the ranked action queue with its reason codes — this is what an editor
   actually sees.
5. (30s) Limitations and the no-go list — what this tool is not allowed to do on its own.

**Social-post cut (one finding + one chart + one method sentence + link):**
> Built a refresh-priority model for 30K+ content pages. A naive train/test split made it look
> nearly perfect (AUC 0.76); grouping the split by client — so no client leaks between train and
> test — brought the honest number down to 0.61, and that honest number is the one that matters.
> Full write-up: [paper URL below].

**Employer-facing summary (what I built · on what data · what it showed):**
I built a decline-risk scoring model and ranked action-playbook for content refresh
prioritization, trained on FlyRank's 30,000-row anonymized content dataset (32 real
pseudonymized clients) plus a month of a 79M-row warehouse release. The model beat a transparent
baseline rule on every metric under an honest, client-grouped validation split, and I explicitly
demonstrated (not just claimed) how much a naive split would have inflated that number. The
output is a ranked, reason-coded queue designed for human-in-the-loop review, with documented
limitations and a no-go list for what should never be automated.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere — only pseudonymized `client_id`/`content_id`
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
